# Derogatory matrices, Positive & Negative definite matrices.

### 1.	Write a python program to accept a square matrix from the user and check if it is derogatory or not.

In [1]:
import numpy as np
from sympy import Symbol
from math import isclose

def echelon_form(mat, pivots=False):
    mat = np.array(mat, dtype=float)
    r, c = mat.shape
    pivot_set = set()
    shift = 0
    
    for i in range(min(r, c)):
        while i+shift<c and all(isclose(elem, 0, abs_tol=1e-08)
                                for elem in mat[i:, i+shift]): shift += 1
        if i+shift==c: break
        
        pivot = mat[i, i+shift]
        for next_row in range(i+1, r):
            below_pivot = mat[next_row, i+shift]
            if not isclose(below_pivot, 0, abs_tol=1e-08):
                if not isclose(pivot, 0, abs_tol=1e-08):
                    mat[next_row] -= (below_pivot/pivot)*mat[i]
                else:
                    mat[[i, next_row]] = mat[[next_row, i]]
                    pivot = below_pivot
        
        pivot_set.add(i+shift)
        mat[i, :i+shift] = 0.
    
    if pivots:
        return mat, pivot_set
    return mat

def solve(mat):
    mat, pivots = echelon_form(mat, pivots=True)
    r, c = mat.shape
    
    if c-1 in pivots:
        return False
    
    A, B = mat[len(pivots)-1::-1, -2::-1], mat[len(pivots)-1::-1, -1]
    x = []
    
    for eq, val in zip(A, B):
        sm = 0
        for i, coeff in enumerate(eq):
            if i==len(x):
                if c-2-i in pivots:
                    x.append((val-sm)/coeff)
                    break
                x.append(Symbol(f'x{c-1-i}'))
            sm += coeff*x[i]
    
    for i in range(c-1-len(x), 0, -1):
        x.append(Symbol(f'x{i}'))
    
    return x[::-1]

In [4]:
import numpy as np
from sympy import Symbol, eye, det, roots, Matrix, Poly, S

def get_A_LI(mat, var='lamda'):
    r, c = mat.shape
    L = Symbol(var)
    I = eye(r)
    
    return mat - L*I

def charac_poly(mat, var='lamda'):
    A_LI = get_A_LI(mat, var=var)
    
    return det(A_LI).as_poly().monic()

def eig_vals(mat):
    charac = charac_poly(mat)
    
    return roots(charac)

def minimal_poly(mat, var='x'):
    r, c = mat.shape
    M = mat
    v = np.random.randn(c)
    
    res = np.column_stack((v, M@v, np.zeros(c)))
    for _ in range(r-1):
        soln = solve(res)
        if not all(val==0 for val in soln):
            coeffs = [val.subs(*val.free_symbols, 1).round(8)
                      for val in reversed(soln)]
            return Poly(coeffs, Symbol(var))
        
        M = M@M
        res = np.insert(res, -1, M@v, axis=1)
    
    return charac_poly(mat, var=var)

def is_derogatory(mat):
    r, c = mat.shape
    vals = eig_vals(mat)
    
    if all(m==1 for m in vals.values()):
        return False
    return minimal_poly(mat).degree()<r

In [5]:
def input_mat(square=False, dtype=float):
    if square:
        r = c = int(input('Enter the dimension of the square matrix: '))
    else:
        r = int(input('Enter the number of rows: '))
        c = int(input('Enter the number of columns: '))
    
    ret_mat = []
    for i in range(r):
        ret_row = [dtype(input(f'Enter element for position [{i+1}, {j+1}]: ')) for j in range(c)]
        ret_mat.append(ret_row)
    
    return ret_mat

A = Matrix(input_mat(square=True, dtype=S))
print('Matrix A:', A)

print('Is A derogatory?:', is_derogatory(A))

Matrix A: Matrix([[1, 0], [0, 1]])
Is A derogatory?: True


### 2.	Write a python program to accept a square matrix from the user and check if it is positive or negative definite.

In [4]:
from sympy import Symbol, eye, det, roots, Matrix, S

def get_A_LI(mat, var='lamda'):
    r, c = mat.shape
    L = Symbol(var)
    I = eye(r)
    
    return mat - L*I

def charac_poly(mat, var='lamda'):
    A_LI = get_A_LI(mat, var=var)
    
    return det(A_LI).as_poly().monic()

def eig_vals(mat):
    charac = charac_poly(mat)
    
    return roots(charac)

def is_positive_definite(mat):
    if mat!=mat.T:
        return False
    
    vals = eig_vals(mat)
    return all(val>0 for val in vals)

def is_negative_definite(mat):
    if mat!=mat.T:
        return False
    
    vals = eig_vals(mat)
    return all(val<0 for val in vals)

In [5]:
def input_mat(square=False, dtype=float):
    if square:
        r = c = int(input('Enter the dimension of the square matrix: '))
    else:
        r = int(input('Enter the number of rows: '))
        c = int(input('Enter the number of columns: '))
    
    ret_mat = []
    for i in range(r):
        ret_row = [dtype(input(f'Enter element for position [{i+1}, {j+1}]: ')) for j in range(c)]
        ret_mat.append(ret_row)
    
    return ret_mat

A = Matrix(input_mat(square=True, dtype=S))
print('Matrix A:', A)

print('Is A positive definite?:', is_positive_definite(A))
print('Is A negative definite?:', is_negative_definite(A))

Matrix A: Matrix([[2, 0], [0, 1]])
Is A positive definite?: True
Is A negative definite?: False
